In [1]:
import pandas as pd
import numpy as np
import torch
import plotly.graph_objects as go
import matplotlib.pyplot as plt

from src.utils import generate_mask_tensor
from src.embedding import embed
from src.gp_ccm import GP_ccm_sig, run_sigGPCCM_experiment, GP_ccm_sig_predict
from src.sp_ccm import run_SP_CCM, SP_CCM_iaaft, run_ccm_experiment
from src.iaaft import surrogates

from scipy.stats import ranksums
torch.set_printoptions(sci_mode = False)

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

Using device: cuda



# Load data

In [3]:
co2_norm = torch.load("data/CO2_vostok_stan_400kyr_timeseries.pt").to(torch.float32)
temp_norm = torch.load("data/TEMP_vostok_stan_400kyr_timeseries.pt").to(torch.float32)

# Less noise

In [138]:
k = 3
N_TRAIN = torch.tensor([200]).to(device)

##############
### sigCCM ###
##############

sig_filter = torch.ones(size = (k, )).to(device)
sig_shift = torch.tensor(sig_filter.shape[0] - 1).to(device) + 2

NOISE_SCALE = torch.tensor([0.05], device = device)
RBF_SCALE = torch.tensor([0.4], device = device)

y_embeddings, x_gt = embed(
    filter = sig_filter, 
    y = co2_norm, 
    x = temp_norm, 
    max_pos_offset = sig_shift, 
    device = device)

rho, nlml, x_test_mean_est, x_test_covar_est = GP_ccm_sig_predict(
    y_embeddings_train = y_embeddings[0 : N_TRAIN].unsqueeze(-1), # [N, E, 1]
    y_embeddings_test = y_embeddings[N_TRAIN : ].unsqueeze(-1), 
    x_train = x_gt[0 : N_TRAIN], # [N]
    x_test = x_gt[N_TRAIN : ], 
    noise = NOISE_SCALE, 
    rbf_sigma = RBF_SCALE, 
    device = device)

print(rho)

tensor(0.5156, device='cuda:0')


In [131]:
k = 4
N_TRAIN = torch.tensor([200]).to(device)

##############
### sigCCM ###
##############

sig_filter = torch.ones(size = (k, )).to(device)
sig_shift = torch.tensor(sig_filter.shape[0] - 1).to(device) + 4
print(sig_shift)

NOISE_SCALE = torch.tensor([0.05], device = device)
RBF_SCALE = torch.tensor([0.22], device = device)

y_embeddings, x_gt = embed(
    filter = sig_filter, 
    y = co2_norm, 
    x = temp_norm, 
    max_pos_offset = sig_shift, 
    device = device)

rho, nlml, x_test_mean_est, x_test_covar_est = GP_ccm_sig_predict(
    y_embeddings_train = y_embeddings[0 : N_TRAIN].unsqueeze(-1), # [N, E, 1]
    y_embeddings_test = y_embeddings[N_TRAIN : ].unsqueeze(-1), 
    x_train = x_gt[0 : N_TRAIN], # [N]
    x_test = x_gt[N_TRAIN : ], 
    noise = NOISE_SCALE, 
    rbf_sigma = RBF_SCALE, 
    device = device)

print(rho)

tensor(7, device='cuda:0')
tensor(0.5229, device='cuda:0')


In [150]:
sym = torch.triu(x_test_covar_est, diagonal = 1).add(torch.triu(x_test_covar_est, diagonal = 1).mT)
# x_test_covar_est_psd = sym.add(torch.eye(n = sym.shape[0], device = device).mul(torch.clip(x_test_covar_est.diag(), min = 0.00001)))
x_test_covar_est_psd = sym.add(torch.eye(n = sym.shape[0], device = device).mul(torch.clip(x_test_covar_est.diag(), min = 0.000).add(0.2)))
# x_test_covar_est_psd_alternative = x_test_covar_est.add(torch.eye(n = sym.shape[0], device = device).mul(0.09))

# Initalise
fig = go.Figure()

fig.add_trace(go.Scatter(x = torch.arange(0, temp_norm.shape[0]), y = temp_norm, # reversing the meaning of x
                        mode = 'lines',
                        name = 'true temperature',
                        line_color = "black"))

fig.add_trace(go.Scatter(x = torch.arange(N_TRAIN.item(), temp_norm.shape[0]), y = x_test_mean_est.cpu(), # reversing the meaning of x
                        mode = 'lines',
                        name = 'mean reconstruction',
                        line_color = 'rgba(0,97,255, 0.9)'))


for i in range(20):
    # Sample via Cholesky
    L = torch.linalg.cholesky(x_test_covar_est_psd.cpu())
    Z = torch.randn(size = [L.shape[-1]], dtype = torch.float32).unsqueeze(0)
    sample = x_test_mean_est.cpu() + torch.matmul(Z, L)  
    fig.add_trace(go.Scatter(x = np.arange(N_TRAIN.item(), co2_norm.shape[0]), y = sample.squeeze(), 
                        mode = 'lines',
                        name = 'Sample',
                        showlegend = False,
                        line = dict(
                                color = 'rgba(0,97,255, 0.3)',
                                width = 0.5
                            )
                        ))

fig.update_layout(template = "simple_white")
fig.update_layout(font_family = "Lato")
fig.update_layout(legend = dict(x = 0.01, y = 0.9, bgcolor = "rgba(0,0,0,0)"))

fig.update_layout(width = 1000, height = 500)
fig.update_layout(yaxis_range = [-6, 8])

fig.show()

# Ensure symmetry and pos

In [ ]:
x_test_covar_est

# CO2 -> temp

In [242]:
k = 7
N_TRAIN = torch.tensor([200]).to(device)

##############
### sigCCM ###
##############

sig_filter = torch.ones(size = (k, )).to(device)
sig_shift = torch.tensor(sig_filter.shape[0] - 1).to(device)
# -1 is towards overlap

NOISE_SCALE = torch.tensor([0.05], device = device)
RBF_SCALE = torch.tensor([2.7], device = device)

y_embeddings, x_gt = embed(
    filter = sig_filter, 
    y = temp_norm, 
    x = co2_norm, 
    max_pos_offset = sig_shift, 
    device = device)

rho, nlml, x_test_mean_est, x_test_covar_est = GP_ccm_sig_predict(
    y_embeddings_train = y_embeddings[0 : N_TRAIN].unsqueeze(-1), # [N, E, 1]
    y_embeddings_test = y_embeddings[N_TRAIN : ].unsqueeze(-1), 
    x_train = x_gt[0 : N_TRAIN], # [N]
    x_test = x_gt[N_TRAIN : ], 
    noise = NOISE_SCALE, 
    rbf_sigma = RBF_SCALE, 
    device = device)

print(rho)

tensor(0.6426, device='cuda:0')


In [248]:
sym = torch.triu(x_test_covar_est, diagonal = 1).add(torch.triu(x_test_covar_est, diagonal = 1).mT)
# x_test_covar_est_psd = sym.add(torch.eye(n = sym.shape[0], device = device).mul(torch.clip(x_test_covar_est.diag(), min = 0.00001)))
x_test_covar_est_psd = sym.add(torch.eye(n = sym.shape[0], device = device).mul(torch.clip(x_test_covar_est.diag(), min = 0.000).add(0.08)))
# x_test_covar_est_psd_alternative = x_test_covar_est.add(torch.eye(n = sym.shape[0], device = device).mul(0.09))

# Initalise
fig = go.Figure()

fig.add_trace(go.Scatter(x = torch.arange(0, temp_norm.shape[0] -7), y = co2_norm, # reversing the meaning of x
                        mode = 'lines',
                        name = 'true temperature',
                        line_color = "black"))

fig.add_trace(go.Scatter(x = torch.arange(N_TRAIN.item(), temp_norm.shape[0]), y = x_test_mean_est.cpu(), # reversing the meaning of x
                        mode = 'lines',
                        name = 'mean reconstruction',
                        line_color = 'rgba(0,97,255, 0.7)'))


for i in range(20):
    # Sample via Cholesky
    L = torch.linalg.cholesky(x_test_covar_est_psd.cpu())
    Z = torch.randn(size = [L.shape[-1]], dtype = torch.float32).unsqueeze(0)
    sample = x_test_mean_est.cpu() + torch.matmul(Z, L)  
    fig.add_trace(go.Scatter(x = np.arange(N_TRAIN.item(), co2_norm.shape[0]), y = sample.squeeze(), 
                        mode = 'lines',
                        name = 'Sample',
                        showlegend = False,
                        line = dict(
                                color = 'rgba(0,97,255, 0.3)',
                                width = 0.5
                            )
                        ))

fig.update_layout(template = "simple_white")
fig.update_layout(font_family = "Lato")
fig.update_layout(legend = dict(x = 0.01, y = 0.9, bgcolor = "rgba(0,0,0,0)"))

fig.update_layout(width = 1000, height = 500)
fig.update_layout(yaxis_range = [-3.2, 3.0])

fig.show()